# Explore the catalog using interactive plotly graphs

In [1]:
import json
import numpy as np
import pandas as pd

import sys, os
from io import StringIO

import matplotlib.pyplot as plt
import plotly.express as px

from pathlib import Path

# Ensure project root is on sys.path so `import paths` finds the top-level paths.py
proj_root = Path('/Users/liekevanson/Documents/Projects/post_mt_review').resolve()
if str(proj_root) not in sys.path:
    sys.path.insert(0, str(proj_root))

from paths import DUMMY_CATALOG, MAIN_CATALOG

file_path = MAIN_CATALOG

In [2]:
# === Load JSON ===
def read_json_file(file_path):
    try:
        with open(file_path, "r") as f:
            data = json.load(f)
        return data
    except Exception as e:
        print(f"Error reading file: {e}")
        return None

# ==  Function to easily extract some of the cols ==    
def extract_array(data, key):
    """Extracts a given key from each system and returns a NumPy array."""
    try:
        values = [entry.get(key, np.nan) for entry in data]
        return np.array(values)
    except Exception as e:
        print(f"Error extracting {key}: {e}")
        return None

def extract_multiple(data, keys):
    return {key: extract_array(data, key) for key in keys}


In [3]:
# Load json data 
data = read_json_file(file_path) #"../data/post_mt_systems.json")
# show_available_keys
# print(list(data[0].keys()))

# extract variables
catalog_data = extract_multiple(data, ["M1", "q", "Period"])
type1 = [entry.get("Type1", "") for entry in data]
type2 = [entry.get("Type2", "Unknown") for entry in data]

print(np.unique(type1))
print(np.unique(type2))

# display(catalog_data[''])

['?' 'B' 'B, Contact' 'B, Contact?' 'B0-1Ve' 'B0-B.05III , Contact'
 'B0.5V, Contact' 'B0V, Contact' 'B1-1.5V, Contact' 'B1.5 V' 'B1V'
 'B1V, Contact' 'B2-3Ve' 'B2.5V' 'B2V' 'B3' 'B3Ve' 'B5V, Contact' 'B6V'
 'B?' 'BIV' 'Ba' 'Be' 'MS' 'MS/WD?' 'MS?' 'O, Contact' 'O-O4 V' 'O-O5.5 V'
 'O-O6 I' 'O-O9' 'O-uncertain' 'O/Be' 'O4 V' 'O4.5 V(n)((fc))z, Contact'
 'O4.5-6V, Contact' 'O5 V' 'O6 V' 'O6-7:(f):, Contact' 'O6.5 V'
 'O6.5-7 Ia, Contact' 'O6.5III' 'O6.5V((f)), Contact' 'O7 III' 'O7 V'
 'O7.5 III' 'O7::' 'O7V' 'O7V, Contact' 'O8V:' 'O9 III:' 'O9 V'
 'O9.5V, Contact' 'O9.7: V:' 'O9II, Contact' 'O9V, Contact' 'O9Ve'
 'ON9 Ia' 'RG' 'SB2' 'barium star' 'dBa' 'gBa' 'sgCH']
['B, Contact' 'B, Contact?' 'B0 III' 'B1-1.5V, Contact'
 'B1-2 IV/V, Contact' 'B1.5III, Contact' 'B1III, Contact' 'B1V, Contact'
 'B2 Ip' 'B2V, Contact' 'B4' 'B5/7V, Contact' 'B7III' 'B:' 'BH' 'BH?'
 'F:6.5:IV:' 'NS' 'NS/WD/MS?' 'NS/WD?' 'O, Contact' 'O4 f +, Contact'
 'O5.5 V(n)((fc))z, Contact' 'O6V((f)), Contact' 'O6V, C

In [4]:
# === Interactive Plotting with Plotly ===
def plotly_vars(col1, col2, catalog_data, data=None, logx=False, logy=False):
    """
    Plot col2 vs. col1 from catalog_data (e.g. output of extract_multiple),
    optionally using metadata (from JSON) for hover and coloring.
    a
    Parameters:
        col1, col2: str
            Keys to plot (e.g. 'M1', 'q')
        catalog_data: dict of str -> np.ndarray
            Typically from extract_multiple()
        data: list of dict
            Raw JSON data for metadata (optional but enables color, hover)
        logx, logy: bool
            Whether to use logarithmic axes
    """
    # Extract central values and uncertainties
    x = catalog_data[col1][:, 1]
    xerr_lo = x - catalog_data[col1][:, 0]
    xerr_hi = catalog_data[col1][:, 2] - x

    y = catalog_data[col2][:, 1]
    yerr_lo = y - catalog_data[col2][:, 0]
    yerr_hi = catalog_data[col2][:, 2] - y

    # Default metadata
    N = len(x)
    system_name = [""] * N
    type1 = [""] * N
    type2 = ["Unknown"] * N
    marker_size = [4] * N

    if data is not None:
        system_name = [entry.get("System Name", "") for entry in data]
        type1 = [entry.get("Type1", "") for entry in data]
        type2 = [entry.get("Type2", "Unknown") for entry in data]
        marker_size = [1 if t2 == "WD" else 5 for t2 in type2]

    fig = px.scatter(
        x=x,
        y=y,
        color=type2,
        hover_data={"System Name": system_name, "Type1": type1},
        error_x=xerr_hi,
        error_x_minus=xerr_lo,
        error_y=yerr_hi,
        error_y_minus=yerr_lo,
        size=marker_size,
        size_max=5,
        labels={"x": col1, "y": col2}
    )

    fig.update_layout(
        width=900,
        height=600,
        xaxis_title=col1,
        yaxis_title=col2,
        xaxis_title_font=dict(size=18),
        yaxis_title_font=dict(size=18),
        legend=dict(font=dict(size=16)),
    )

    fig.update_yaxes(tickfont=dict(size=14))
    fig.update_xaxes(tickfont=dict(size=14))

    if logx:
        fig.update_xaxes(type="log")
    if logy:
        fig.update_yaxes(type="log")

    return fig


In [5]:
# Load json data 
data = read_json_file(file_path) #"../data/post_mt_systems.json")
# show_available_keys
print(list(data[0].keys()))

# extract variables
catalog_data = extract_multiple(data, ["M1", "q", "Period"])


['System Name', 'Type1', 'Type2', 'Detection Method', 'Reference', 'Notes', 'RA', 'Dec', 'Period', 'Eccentricity', 'M1', 'M1_sin3i', 'M2', 'M2_sin3i', 'q', 'Mass Function', 'class', 'Simbad']


In [6]:
# Plot q vs M1
plotly_vars("q", "Period", catalog_data, data=data, logy=True)


TypeError: unsupported operand type(s) for -: 'NoneType' and 'NoneType'